# Flow Matching for Discrete Data: Sudoku

Two methods applied to *Sudoku Extreme* (9×9 grid, 81-token context),
sharing the same 28M-parameter DiT backbone.

We model discrete sequences $\mathbf{y} = (y^1, \ldots, y^L) \in V^L$ over a
vocabulary of size $|V|$. Each token $k$ has a learned embedding
$\mathbf{w}_k \in S^{d-1} \subset \mathbb{R}^d$ (a unit vector), stored as a column
of $W_E \in \mathbb{R}^{d \times |V|}$.

1. **vMF (spherical flow matching).** $d{=}11$. The forward process is
   $p_t(\mathbf{h}\mid \mathbf{w}_k) = f(\mathbf{h};\,\mathbf{w}_k,\,\kappa(t))$ with the von
   Mises–Fisher density and a monotone schedule $\kappa(t)$ from $\kappa(0){=}0$
   (uniform on $S^{d-1}$) to $\kappa(1){=}\kappa_{\max}$. We provide **two
   checkpoints** that differ only in training:
   - `vmf_tc_d11_p1` — the backbone is given the current noise level
     $\kappa(t)/\kappa_{\max}$ via adaLN. This matches the paper's setup
     and is the *mathematically correct* parameterization: the conditional
     velocity field on $S^{d-1}$ depends on $\kappa(t)$, so the predictor
     should see it.
   - `vmf_d11_p1` — same loss, but the noise level is **not** passed in.
     The backbone has to infer $\kappa(t)$ from $\mathbf{h}_t$ itself. Strictly
     speaking this is under-specified (different $\kappa$ can produce
     similar $\mathbf{h}_t$), but in practice the model can still learn to do it.
     We include it as an ablation.
2. **Masked diffusion** ($p=1$). Discrete CTMC baseline (MDLM-style): tokens
   are masked i.i.d. with probability $t$ and the model predicts the original
   vocabulary; cross-entropy is computed only on masked positions. The reverse
   kernel unmasks tokens progressively at sampling time.

So: **two methods** (continuous vMF on the sphere, discrete masked CTMC),
**three checkpoints** to play with — `vmf_d11_p1`, `vmf_tc_d11_p1`, `masked_p1`.

The notebook is inference-only. Reference: *Spherical Flows for Sampling
Discrete Distributions* ([arXiv:2605.05629](https://arxiv.org/abs/2605.05629)).
The model + sampler code under `flows_categorical/` is copied (with attribution)
from the paper's source repo.


## 0. Setup

Run this once. On Colab it clones the tutorial repo so `flows_categorical/` is
importable, and installs the few extra dependencies. Local users can skip the
clone if they already have the repo.


In [ ]:
# --- Colab setup ---
import os, sys, subprocess

REPO_URL = "https://github.com/JChemseddine/fm_tutorial.git"
REPO_DIR = "fm_tutorial"

ON_COLAB = "google.colab" in sys.modules
if ON_COLAB and not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
elif os.path.isdir("flows_categorical"):
    pass  # already at repo root
elif os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)

# Extra deps not in requirements.txt
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "einops", "huggingface_hub", "safetensors"])

# Make `flows_categorical` importable
sys.path.insert(0, os.getcwd())
print("CWD:", os.getcwd())
print("flows_categorical present:", os.path.isdir("flows_categorical"))


In [ ]:
import os
import json

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Determinism for the tutorial — same puzzles, same samples for everyone.
SEED = 1234   # also the paper-reproducibility seed for the Sudoku checkpoints
torch.manual_seed(SEED)
np.random.seed(SEED)


## 1. Locate pretrained checkpoints

The three checkpoints are ~440 MB each (28M params + Adam state + EMA copy).
Two ways to get them:

- **Local (developer machine)** — drop `final.pt` files into `networks/<method>/`,
  e.g. `networks/vmf_d11_p1/final.pt`. The cell below detects this and skips the
  HF download.
- **Hugging Face Hub** — they live at
  [`Jugc/fm-tutorial`](https://huggingface.co/Jugc/fm-tutorial) with layout

  ```
  Jugc/fm-tutorial/
  ├── vmf_d11_p1/checkpoint.pt
  ├── vmf_tc_d11_p1/checkpoint.pt
  └── masked_p1/checkpoint.pt
  ```

  The notebook auto-falls-back to HF Hub if `networks/` is missing.


In [ ]:
METHOD_NAMES = ["vmf_d11_p1", "vmf_tc_d11_p1", "masked_p1"]
HF_REPO = "Jugc/fm-tutorial"

def resolve_checkpoint(name):
    """Return a local checkpoint path. Prefers ./networks/<name>/final.pt;
    falls back to downloading <name>/checkpoint.pt from HF Hub."""
    local = os.path.join("networks", name, "final.pt")
    if os.path.exists(local):
        return local
    # Download from HF Hub
    return hf_hub_download(repo_id=HF_REPO, filename=f"{name}/checkpoint.pt", repo_type="model")

CKPT_PATHS = {name: resolve_checkpoint(name) for name in METHOD_NAMES}
for name, p in CKPT_PATHS.items():
    src = "local" if p.startswith("networks/") else "HF Hub"
    print(f"  {name:18s}  [{src}]  {p}")


In [ ]:
# Held-out puzzles: bundled in the tutorial repo (100 random puzzles from sudoku-extreme test split)
PUZZLES_PATH = "data/sudoku_extreme_100.npz"
data = np.load(PUZZLES_PATH)
test_inputs = torch.from_numpy(data["inputs"].astype(np.int64))   # (100, 81), 0=blank, 1-9=clue
test_labels = torch.from_numpy(data["labels"].astype(np.int64))   # (100, 81), 1-9 (full solution)

assert test_inputs.min() >= 0 and test_inputs.max() <= 9
assert test_labels.min() >= 1 and test_labels.max() <= 9

print(f"Loaded {len(test_inputs)} test puzzles, shape {tuple(test_inputs.shape)}")
print(f"Avg clues per puzzle: {(test_inputs > 0).float().sum(dim=1).mean():.1f}  "
      f"(min {(test_inputs > 0).sum(dim=1).min().item()}, max {(test_inputs > 0).sum(dim=1).max().item()})")


## 2. Anatomy of a Sudoku puzzle

Each puzzle is a flat sequence of length 81 (the 9×9 grid in row-major order).
Blanks are token 0; the model must predict tokens 1–9 for those positions.
*Sudoku Extreme* puzzles have as few as 17 clues — they are deliberately hard.


In [ ]:
def show_grid(tokens, clue_mask=None, gt=None, title=None, ax=None):
    '''Plot a (81,) tensor as a 9x9 grid.
    - clue cells: bold black
    - non-clue cells: blue if they match `gt` (when provided), red if wrong
    - if `gt` is None, all non-clue cells are blue (no correctness signal)
    '''
    if ax is None:
        _, ax = plt.subplots(figsize=(3.2, 3.2))
    grid = tokens.view(9, 9).cpu().numpy()
    ax.set_xlim(0, 9); ax.set_ylim(9, 0)
    ax.set_xticks([]); ax.set_yticks([])
    for x in range(10):
        lw = 2.0 if x % 3 == 0 else 0.5
        ax.plot([x, x], [0, 9], "k-", lw=lw)
        ax.plot([0, 9], [x, x], "k-", lw=lw)
    cm = None if clue_mask is None else clue_mask.view(-1).cpu().numpy()
    gt_flat = None if gt is None else gt.view(-1).cpu().numpy()
    tok_flat = tokens.view(-1).cpu().numpy()
    for i in range(9):
        for j in range(9):
            idx = i * 9 + j
            v = int(grid[i, j])
            if v == 0:
                continue
            is_clue = cm is not None and bool(cm[idx])
            if is_clue:
                color, weight = "black", "bold"
            elif gt_flat is not None and int(tok_flat[idx]) != int(gt_flat[idx]):
                color, weight = "#d62728", "normal"   # red for wrong
            else:
                color, weight = "#1f77b4", "normal"   # blue for correct (or no gt)
            ax.text(j + 0.5, i + 0.5, str(v), ha="center", va="center",
                    fontsize=14, fontweight=weight, color=color)
    if title:
        ax.set_title(title, fontsize=10)
    return ax


idx = 0
clue_mask = (test_inputs[idx] != 0)
fig, axes = plt.subplots(1, 2, figsize=(6.4, 3.2))
show_grid(test_inputs[idx], clue_mask, title="Puzzle (clues bold)", ax=axes[0])
show_grid(test_labels[idx], clue_mask, title="Ground-truth solution", ax=axes[1])
plt.tight_layout(); plt.show()


## 3. Loading the three methods

Each checkpoint ships with the training `config.json` and a `checkpoint.pt`
holding the model weights and (for vMF) the learned $\kappa(t)$ schedule parameters.
A small helper hides the boilerplate.


In [ ]:
from flows_categorical.config import Config
from flows_categorical.model.backbone import ContinuousTransformer, MaskedTransformer
from flows_categorical.methods import create_sampler
from flows_categorical.methods.continuous.spherical.vmf import PsiTable
from flows_categorical.schedule.cdcd_warp import CDCDWarp


def load_method(name, device=DEVICE, use_ema=True):
    """Load (config, model, sampler) for a method name. Uses EMA weights by default."""
    ckpt_path = CKPT_PATHS[name]
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    config = Config.from_dict(ckpt["config"])

    # Backbone
    if config.flow.noise_process == "masked":
        model = MaskedTransformer(config.model.vocab_size, config.model)
    else:
        model = ContinuousTransformer(config.model.vocab_size, config.model)

    # Pick EMA if asked & available; else raw model_state_dict
    src_sd = ckpt.get("ema_state_dict") if use_ema else None
    if src_sd is None:
        src_sd = ckpt["model_state_dict"]
        weight_src = "model_state_dict"
    else:
        weight_src = "ema_state_dict"
    msd = {k.removeprefix("_orig_mod."): v for k, v in src_sd.items()}
    missing, unexpected = model.load_state_dict(msd, strict=False)
    if missing:
        print(f"  [{name}] missing keys: {missing[:3]}{'...' if len(missing) > 3 else ''}")
    if unexpected:
        print(f"  [{name}] unexpected keys: {unexpected[:3]}{'...' if len(unexpected) > 3 else ''}")
    model.to(device).eval()

    # Sampler-side extras
    sampler_kwargs = {}
    if config.flow.noise_process == "vmf":
        sampler_kwargs["psi_table"] = PsiTable(
            config.model.embed_dim,
            kappa_range=config.flow.kappa_max,
            grid_size=config.flow.psi_grid_size,
        )
    if config.flow.use_warp and "warp_state" in ckpt:
        warp = CDCDWarp(
            kappa_max=config.flow.kappa_max,
            num_bins=config.flow.warp_bins,
            warmup_steps=config.flow.time_warp_warmup,
            ema_decay=config.flow.warp_ema_decay,
            noise_increasing=False,  # vMF: parameter (kappa) increases with signal
        )
        warp.load_state_dict(ckpt["warp_state"])
        warp.to(device).eval()
        sampler_kwargs["warp"] = warp

    sampler = create_sampler(model, config, **sampler_kwargs)
    print(f"  [{name}] loaded from {weight_src}  (step {ckpt.get('step','?')})")
    return config, model, sampler


methods = {}
for name in METHOD_NAMES:
    print(f"Loading {name}...")
    methods[name] = load_method(name)

# Quick model-size summary
print()
for name, (cfg, mdl, _) in methods.items():
    n_params = sum(p.numel() for p in mdl.parameters())
    tc = "with t-cond" if cfg.model.time_conditioning else "no t-cond"
    print(f"  {name:18s}  {n_params/1e6:5.1f}M params  noise={cfg.flow.noise_process:6s} d={cfg.model.embed_dim:3d}  {tc}")


## 4. Solving one puzzle with each method

Same puzzle, same clues. The model conditions on the clue positions via
`clue_mask` (which positions are observed) and `clue_values` (the digits at
those positions). The sampler pins the clue embeddings throughout the trajectory.


In [ ]:
torch.manual_seed(SEED)

def sample_one(sampler, puzzle, num_samples=1, return_intermediates=False):
    """Sample completions for a single puzzle. Returns (tokens, intermediates)."""
    puzzle = puzzle.to(DEVICE)
    clue_mask = (puzzle != 0).unsqueeze(0).expand(num_samples, -1).contiguous()
    clue_values = puzzle.unsqueeze(0).expand(num_samples, -1).contiguous()
    out = sampler.sample(
        num_samples=num_samples, device=DEVICE,
        clue_mask=clue_mask, clue_values=clue_values,
        return_intermediates=return_intermediates, verbose=False,
    )
    return out  # dict with 'tokens', possibly 'intermediates'


idx = 0
puzzle = test_inputs[idx]
clue_mask = (puzzle != 0)
gt = test_labels[idx]

results = {}
for name, (_, _, sampler) in methods.items():
    out = sample_one(sampler, puzzle, num_samples=1)
    results[name] = out["tokens"][0].cpu()

fig, axes = plt.subplots(1, 4, figsize=(12.8, 3.2))
show_grid(puzzle, clue_mask, title="Puzzle", ax=axes[0])
for ax, (name, tokens) in zip(axes[1:], results.items()):
    correct = (tokens == gt).float().mean().item() * 100
    show_grid(tokens, clue_mask, gt=gt, title=f"{name}\n{correct:.0f}% cell-correct", ax=ax)
plt.tight_layout(); plt.show()


## Sampler knobs

For the spherical (vMF) sampler we use a predictor–corrector ODE on $S^{d-1}$.
Each predictor step is one Euler step along the vMF-derived velocity; an
optional Langevin corrector then takes `CORRECTOR_STEPS` score-based moves
on the tangent plane, applied every `CORRECTOR_INTERVAL` predictor steps.

$$\mathrm{NFE} \;=\; n_{\mathrm{pred}}\;+\;\Big\lfloor \tfrac{n_{\mathrm{pred}}}{n_{\mathrm{int}}} \Big\rfloor \cdot k_{\mathrm{corrector}}.$$

Paper sweeps at total NFE=128 (with interval $n_{\mathrm{int}}=1$):
$(n_{\mathrm{pred}}, k_{\mathrm{corrector}}) \in \{(64,1), (32,3), (16,7)\}$.

| Knob | What it does |
|------|--------------|
| `PREDICTOR_STEPS` ($n_{\mathrm{pred}}$) | Number of Euler ODE steps along the velocity field. |
| `CORRECTOR_STEPS` ($k_{\mathrm{corrector}}$) | Langevin corrector steps per correction (vMF only). `0` = plain ODE. |
| `CORRECTOR_INTERVAL` ($n_{\mathrm{int}}$) | Apply the corrector every $n_{\mathrm{int}}$-th predictor step. `1` = every step. |
| `CORRECTOR_EPS` | Langevin step size $\varepsilon$. |

The masked sampler has no corrector — only `PREDICTOR_STEPS` (= total NFE for that method) is in scope. The mask-rate schedule $t^p$ is set at training time and is not tunable at inference.


In [ ]:
torch.manual_seed(SEED)

# --- Edit me ---
PREDICTOR_STEPS    = 32   # ODE Euler steps along the velocity field
CORRECTOR_STEPS    = 1    # Langevin corrector steps per correction (vMF only; 0 = plain ODE)
CORRECTOR_INTERVAL = 1    # apply corrector every N-th predictor step (1 = every step)
CORRECTOR_EPS      = 0.01 # Langevin step size eps
# -----------------

n_corrections = PREDICTOR_STEPS // CORRECTOR_INTERVAL
vmf_nfe = PREDICTOR_STEPS + n_corrections * CORRECTOR_STEPS
print(f"vMF: {PREDICTOR_STEPS} predictor + {n_corrections}*{CORRECTOR_STEPS} corrector = {vmf_nfe} NFE")
print(f"masked: {PREDICTOR_STEPS} NFE")

# vMF samplers: all four knobs
for name in ("vmf_d11_p1", "vmf_tc_d11_p1"):
    _, _, sampler = methods[name]
    sampler.num_steps          = PREDICTOR_STEPS
    sampler.corrector_steps    = CORRECTOR_STEPS
    sampler.corrector_interval = CORRECTOR_INTERVAL
    sampler.corrector_epsilon  = CORRECTOR_EPS
    sampler.method = "pc_softmax"  # ODE predictor + Langevin corrector (collapses to plain ODE when CORRECTOR_STEPS=0)

# Masked: only predictor steps apply
methods["masked_p1"][2].num_steps = PREDICTOR_STEPS

fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
show_grid(puzzle, clue_mask, title="Puzzle", ax=axes[0])
for ax, name in zip(axes[1:], methods):
    _, _, sampler = methods[name]
    tokens = sample_one(sampler, puzzle)["tokens"][0].cpu()
    acc = (tokens == gt).float().mean().item() * 100
    nfe_here = vmf_nfe if name != "masked_p1" else PREDICTOR_STEPS
    show_grid(tokens, clue_mask, gt=gt, title=f"{name}\nNFE={nfe_here}  acc={acc:.0f}%", ax=ax)
plt.tight_layout(); plt.show()


## Predictor–corrector vs plain ODE at the same NFE budget

The Langevin corrector spends extra forward passes refining each predictor
step. At equal total NFE, is it worth it? Two settings with the same total
budget $N$:

- **Plain ODE** — all NFE goes into predictor steps: $(n_{\mathrm{pred}}, k) = (N, 0)$.
- **PC** — half predictor, one corrector per predictor: $(n_{\mathrm{pred}}, k) = (N/2,\, 1)$, NFE $= N/2 + N/2 = N$.

Same puzzle, both vMF checkpoints. The masked sampler has no corrector, so it
is not included here.


In [ ]:
torch.manual_seed(SEED)

# --- Edit me ---
TOTAL_NFE = 32
# -----------------

def run_with(sampler, predictor_steps, corrector_steps, corrector_interval=1, corrector_eps=0.01):
    sampler.num_steps          = predictor_steps
    sampler.corrector_steps    = corrector_steps
    sampler.corrector_interval = corrector_interval
    sampler.corrector_epsilon  = corrector_eps
    sampler.method = "pc_softmax"
    return sample_one(sampler, puzzle)["tokens"][0].cpu()


# Two settings at the same total NFE
ode_pred = TOTAL_NFE
pc_pred  = TOTAL_NFE // 2
pc_corr  = 1
settings = [("ODE", ode_pred, 0), ("PC", pc_pred, pc_corr)]

vmf_names = ("vmf_d11_p1", "vmf_tc_d11_p1")
fig, axes = plt.subplots(2, 3, figsize=(10, 6.4))
for row, name in enumerate(vmf_names):
    show_grid(puzzle, clue_mask, title=name, ax=axes[row, 0])
    _, _, sampler = methods[name]
    for col, (label, pred, corr) in enumerate(settings, start=1):
        tokens = run_with(sampler, pred, corr)
        acc = (tokens == gt).float().mean().item() * 100
        nfe = pred + pred * corr  # interval=1
        show_grid(tokens, clue_mask, gt=gt,
                  title=f"{label}  (n_pred={pred}, k={corr})\nNFE={nfe}  acc={acc:.0f}%",
                  ax=axes[row, col])
plt.tight_layout(); plt.show()


## 6. Visualizing the trajectory

Three views of how the model converges on a solution. All three use a single
puzzle and the spherical (vMF) sampler with `return_intermediates=True` so we
record the model state at every step.


In [ ]:
torch.manual_seed(SEED)

@torch.no_grad()
def step_logits(model, h_or_x, sigma=None, is_continuous=True):
    '''Compute (1, 81, V) logits from a trajectory snapshot.

    For time-conditioned continuous models, pass `sigma = kappa_t / kappa_max`
    (a (B,) tensor); for non-time-conditioned, pass None.
    '''
    inp = h_or_x.to(DEVICE)
    if not is_continuous:
        return model(inp, clue_mask=None)
    h_prime = model(inp, clue_mask=None, sigma=sigma)
    return model.compute_logits(h_prime, W_E=model.get_W_E())


def sigma_for_snap(snap, cfg, batch_size=1):
    '''Return the sigma tensor a time-conditioned sampler would have passed at this snapshot,
    or None for models trained without time conditioning.'''
    if not cfg.model.time_conditioning:
        return None
    kappa_t = snap.get("kappa", None)
    if kappa_t is None:
        return None
    return torch.full((batch_size,), kappa_t / cfg.flow.kappa_max, device=DEVICE)


# Trace one sample with vmf_tc (the time-conditioned, mathematically correct vMF model).
trace_name = "vmf_tc_d11_p1"
cfg_v, mdl_v, sampler_v = methods[trace_name]
sampler_v.num_steps          = 32
sampler_v.method             = "pc_softmax"
sampler_v.corrector_steps    = 1
sampler_v.corrector_interval = 1
out_v = sample_one(sampler_v, puzzle, num_samples=1, return_intermediates=True)
trace = out_v["intermediates"]
print(f"{len(trace)} snapshots from {trace_name}  (kappa: {trace[0]['kappa']:.2f} -> {trace[-1]['kappa']:.2f})")


### Viz 1 — argmax digit + confidence at six snapshots in time

For each cell we plot the model's currently-favoured digit, with alpha set to
the max softmax probability. Early on the grid is washed-out (low confidence,
uniform-ish); by the end the model has committed.


In [ ]:
def viz_argmax_strip(trace, model, cfg, puzzle, n_panels=6):
    is_continuous = (cfg.flow.noise_process != "masked")
    clue_mask_flat = (puzzle != 0)
    idxs = np.linspace(0, len(trace) - 1, n_panels).astype(int)
    fig, axes = plt.subplots(1, n_panels, figsize=(2.2 * n_panels, 2.4))
    for ax, k in zip(axes, idxs):
        snap = trace[k]
        h_or_x = snap.get("h_t", snap.get("x_t"))
        sigma = sigma_for_snap(snap, cfg) if is_continuous else None
        logits = step_logits(model, h_or_x, sigma=sigma, is_continuous=is_continuous)
        # restrict to digit tokens 1-9 (drop blank slot 0)
        d_probs = F.softmax(logits[0, :, 1:10], dim=-1).cpu()
        confidence, argmax_d = d_probs.max(dim=-1)
        argmax = argmax_d + 1  # back to 1-9
        ax.set_xlim(0, 9); ax.set_ylim(9, 0); ax.set_xticks([]); ax.set_yticks([])
        for x in range(10):
            lw = 1.5 if x % 3 == 0 else 0.3
            ax.plot([x, x], [0, 9], "k-", lw=lw); ax.plot([0, 9], [x, x], "k-", lw=lw)
        for i in range(9):
            for j in range(9):
                v = int(argmax[i * 9 + j])
                a = float(confidence[i * 9 + j])
                is_clue = bool(clue_mask_flat[i * 9 + j])
                ax.text(j + 0.5, i + 0.5, str(v), ha="center", va="center",
                        fontsize=10, alpha=1.0 if is_clue else a,
                        color="black" if is_clue else "#1f77b4",
                        fontweight="bold" if is_clue else "normal")
        t = snap.get("time", k / max(1, len(trace) - 1))
        ax.set_title(f"t={t:.2f}", fontsize=9)
    plt.tight_layout(); plt.show()


viz_argmax_strip(trace, mdl_v, cfg_v, puzzle)


### Viz 2 — per-cell entropy heatmap

At each snapshot we compute the entropy of the per-cell predictive distribution
over the 9 digits, $H(p_i) = -\sum_v p_i(v) \log p_i(v)$. High entropy = the model
is uncertain about cell $i$; near 0 = the model has committed. Clue cells stay at 0.


In [ ]:
def viz_entropy_strip(trace, model, cfg, puzzle, n_panels=6):
    is_continuous = (cfg.flow.noise_process != "masked")
    clue_mask_grid = (puzzle != 0).view(9, 9).numpy()
    idxs = np.linspace(0, len(trace) - 1, n_panels).astype(int)
    fig, axes = plt.subplots(1, n_panels, figsize=(2.2 * n_panels, 2.4))
    log9 = np.log(9.0)
    for ax, k in zip(axes, idxs):
        snap = trace[k]
        h_or_x = snap.get("h_t", snap.get("x_t"))
        sigma = sigma_for_snap(snap, cfg) if is_continuous else None
        logits = step_logits(model, h_or_x, sigma=sigma, is_continuous=is_continuous)
        p = F.softmax(logits[0, :, 1:10], dim=-1)
        H = -(p * p.clamp_min(1e-12).log()).sum(dim=-1).cpu().numpy() / log9
        H = H.reshape(9, 9)
        H[clue_mask_grid] = 0.0
        ax.imshow(H, cmap="viridis", vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])
        t = snap.get("time", k / max(1, len(trace) - 1))
        ax.set_title(f"t={t:.2f}", fontsize=9)
    plt.tight_layout(); plt.show()


viz_entropy_strip(trace, mdl_v, cfg_v, puzzle)


## 7. Solving a batch + validity check

For each of `N_PUZZLES` test puzzles we generate one completion with each
method, then check three things:

1. **Cell accuracy** — fraction of non-clue cells matching the ground-truth solution.
2. **Strict-match** — all 81 cells correct.
3. **Sudoku validity** — every row, column, and 3×3 box contains digits 1–9 exactly once.

Validity is a stricter signal than cell accuracy: a method can get 95% of cells
right and still produce an invalid grid.


In [ ]:
torch.manual_seed(SEED)

def is_valid_sudoku(grid_81):
    g = grid_81.view(9, 9)
    target = torch.arange(1, 10)
    for i in range(9):
        if not torch.equal(torch.sort(g[i]).values, target): return False
        if not torch.equal(torch.sort(g[:, i]).values, target): return False
    for bi in range(3):
        for bj in range(3):
            box = g[3*bi:3*bi+3, 3*bj:3*bj+3].flatten()
            if not torch.equal(torch.sort(box).values, target): return False
    return True


N_PUZZLES = 8  # bump up for a sturdier estimate; ~seconds per puzzle on T4

stats = {name: dict(cell_acc=[], strict=[], valid=[]) for name in methods}
for idx in range(N_PUZZLES):
    puzzle = test_inputs[idx]
    gt = test_labels[idx]
    clue_mask = (puzzle != 0)
    non_clue = ~clue_mask
    for name, (_, _, sampler) in methods.items():
        tokens = sample_one(sampler, puzzle, num_samples=1)["tokens"][0].cpu()
        cell_acc = (tokens[non_clue] == gt[non_clue]).float().mean().item()
        strict = bool((tokens == gt).all())
        valid = is_valid_sudoku(tokens)
        stats[name]["cell_acc"].append(cell_acc)
        stats[name]["strict"].append(strict)
        stats[name]["valid"].append(valid)

print(f"Results over {N_PUZZLES} held-out Sudoku-Extreme puzzles:")
print(f"{'method':<14s}  cell-acc   strict   valid")
for name, d in stats.items():
    print(f"{name:<14s}  {np.mean(d['cell_acc'])*100:6.1f}%   "
          f"{np.mean(d['strict'])*100:5.1f}%   {np.mean(d['valid'])*100:5.1f}%")


## How are these models trained?

This notebook is inference-only, but the relevant equations from the paper:

### vMF / vMF-tc

The conditional probability path on $S^{d-1}$ is
$$p_t(\mathbf{h}\mid \mathbf{w}_k) \;=\; f\!\big(\mathbf{h};\, \mathbf{w}_k,\, \kappa(t)\big)
\;=\; C_d(\kappa(t))\,\exp\!\big(\kappa(t)\,\mathbf{w}_k^\top \mathbf{h}\big),$$
with $C_d(\kappa) = \kappa^{d/2-1} / \big((2\pi)^{d/2}\, I_{d/2-1}(\kappa)\big)$
and $\kappa(t)$ a learned monotone schedule. At $\kappa = 0$ this is uniform on
$S^{d-1}$; as $\kappa \to \infty$ it concentrates on $\mathbf{w}_k$.

For uniform prior $p(k)$ over the vocabulary, the posterior is the softmax
with scale $\kappa$ (Lemma 1 in the paper):
$$p(k \mid \mathbf{h}) \;=\;
\frac{\exp(\kappa\,\mathbf{w}_k^\top \mathbf{h})}{\sum_j \exp(\kappa\,\mathbf{w}_j^\top \mathbf{h})}.$$

This means the backbone only has to predict a unit-norm direction
$\hat{\mathbf{h}}' \in S^{d-1}$; the *logits* at every position are
$\ell_k = \kappa(t)\,\mathbf{w}_k^\top \hat{\mathbf{h}}'$, and **the loss is the standard
position-wise cross-entropy** between those logits and the true tokens:
$$\mathcal{L}_{\mathrm{CE}} \;=\; -\,\mathbb{E}_{t, x_0, \mathbf{h}_t}\!\Big[\,\sum_{i=1}^{L} \log p_\theta(x_0^i \mid \mathbf{h}_t^i)\,\Big].$$

In pseudo-code (one optimizer step):

```python
x_0 = batch                                   # (B, L)  clean token IDs in {1..V}
W_E = normalize(model.W_E, dim=0)             # (d, V)  unit-norm columns w_k in S^{d-1}

t   = uniform(0, 1, size=B)                   # one noise level per sample
k_t = kappa_schedule(t)                       # (B,)    learned monotone kappa(t) >= 0

h_clean = W_E[:, x_0].T                       # (B, L, d)  target embeddings w_{x_0}
h_t     = sample_vmf(mean=h_clean, kappa=k_t) # (B, L, d)  noised states on S^{d-1}

h_hat   = normalize(model.backbone(h_t, t), dim=-1)   # (B, L, d)  predicted direction
logits  = k_t * (h_hat @ W_E)                 # (B, L, V)  l_k = kappa(t) * <w_k, h_hat>

loss    = cross_entropy(logits.flatten(0, 1), x_0.flatten())
loss.backward(); optimizer.step()
```

The schedule $\kappa(t)$ is itself learned (CDCD-style piecewise-linear warp,
fit to the empirical loss curve). `vmf_tc` additionally feeds $\kappa(t)/\kappa_{\max}$
into the backbone via adaLN modulation; the plain `vmf` model gets no time
information and has to infer the noise level from $\mathbf{h}_t$ itself.

The drift, Riemannian score, and reverse-SDE drift are all posterior-weighted
tangent sums in $\mathbf{w}_k$ — they reuse the *same* softmax probabilities computed
during the forward pass (Section 3.4 of the paper). This is how
`pc_softmax` (the predictor–corrector ODE sampler in this notebook) gets both
predictor and corrector steps from a single backbone evaluation per step.

### Masked

Same skeleton with a different noise step and a masked-position loss:

```python
x_0  = batch                                  # (B, L)
t    = uniform(0, 1, size=B)
mask = bernoulli(probs=t.unsqueeze(-1).expand_as(x_0))   # 1 = mask this position
y_t  = where(mask, MASK_TOKEN, x_0)           # corrupted input
logits = model.backbone(y_t)                  # (B, L, V)

loss = cross_entropy(logits[mask], x_0[mask]) # CE only on masked positions
```

At sampling time, the reverse CTMC kernel unmasks tokens progressively from
$t=1$ down to $t \approx 0$ (mask rate $= t^p$ with $p=1$ in our checkpoint).
This is the MDLM line, [Sahoo et al., 2024](https://github.com/kuleshov-group/mdlm).

Full training code lives in the source repo
[`JChemseddine/spherical`](https://github.com/JChemseddine/spherical) (branch
`paper-release-anon`). Each Sudoku checkpoint took ~1M steps at batch 128.


## Things to try

- **Sweep predictor / corrector split.** Try `(PREDICTOR_STEPS, CORRECTOR_STEPS) = (64, 1)`, `(32, 3)`, `(16, 7)` — same total NFE, different time/score ratio.
- **Disable the corrector.** Set `CORRECTOR_STEPS = 0` to get a plain ODE sampler — the Langevin corrector trades wall time for fewer constraint violations.
- **Different seeds.** Wrap the sampling cells in `torch.manual_seed(...)`. Two methods disagreeing on a particular cell is informative.
- **Custom puzzle.** Build your own 81-token puzzle (`0` = blank, `1..9` = clue) and feed it through the methods.
- **vmf vs vmf_tc head-to-head.** Run the batch eval a few times and see whether time-conditioning helps on this dataset.

---

The masked-diffusion sampler is based on
[MDLM (Sahoo et al., 2024)](https://github.com/kuleshov-group/mdlm); the DiT
backbone follows Peebles & Xie 2023. See the source repo for full attributions.
